In [ ]:
import cv2
import numpy as np
import json
import datetime
from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import Video, display
from ipywidgets import widgets
import io

# Initialize YOLOv8 model
yolo_model = YOLO("yolov8n.pt")

# Cricket object classes
CRICKET_CLASSES = {
    'ball': 32,       # sports ball
    'bat': 39,        # baseball bat
    'player': 0,      # person
    'stumps': 9       # bench (placeholder for stumps)
}

def estimate_3d_coordinates(x2d, y2d, frame_w, frame_h, obj_type):
    """Estimate simple 3D coordinates from 2D image points."""
    x_norm = x2d / frame_w
    y_norm = y2d / frame_h

    if obj_type == 'ball':
        z = 0.5 + (1 - y_norm) * 2
        x = (x_norm - 0.5) * 3
        y = (0.5 - y_norm) * 3
    elif obj_type == 'bat':
        z = 0.3 + (1 - y_norm) * 0.7
        x = (x_norm - 0.5) * 2
        y = (0.5 - y_norm) * 1.5
    else:  # players and stumps
        z = 0.1 + (1 - y_norm) * 0.4
        x = (x_norm - 0.5) * 2
        y = 0

    return round(x, 2), round(y, 2), round(z, 2)

# Upload widget
video_upload = widgets.FileUpload(
    accept='.mp4,.avi,.mov',
    multiple=False,
    description='Upload Video'
)

# Output and progress bar
output_area = widgets.Output()
progress_bar = widgets.FloatProgress(value=0, min=0, max=100, description='Processing:')

# Display
display(video_upload)
display(progress_bar)
display(output_area)

def handle_video_upload(change):
    if not video_upload.value:
        return

    with output_area:
        output_area.clear_output()
        print("Processing video...")

        # Get uploaded video content
        video_filename = next(iter(video_upload.value))
        video_content = video_upload.value[video_filename]['content']

        # Save to a temporary file
        temp_video_path = 'uploaded_video.mp4'
        with open(temp_video_path, 'wb') as f:
            f.write(video_content)

        # Read video
        cap = cv2.VideoCapture(temp_video_path)
        frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        ball_points = []
        bat_points = []
        player_points = []
        stump_points = []

        frame_idx = 0
        progress_bar.max = total_frames

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            progress_bar.value = frame_idx

            if frame_idx % 5 == 0:
                detections = yolo_model(frame)[0]

                for box in detections.boxes:
                    x1, y1, x2, y2 = map(float, box.xyxy[0])
                    cls_id = int(box.cls)
                    confidence = float(box.conf)
                    x_center, y_center = (x1 + x2) / 2, (y1 + y2) / 2

                    detected_class = None
                    if cls_id == CRICKET_CLASSES['ball'] and confidence > 0.3:
                        detected_class = 'ball'
                    elif cls_id == CRICKET_CLASSES['bat'] and confidence > 0.3:
                        detected_class = 'bat'
                    elif cls_id == CRICKET_CLASSES['player'] and confidence > 0.5:
                        detected_class = 'player'
                    elif cls_id == CRICKET_CLASSES['stumps'] and confidence > 0.4:
                        detected_class = 'stumps'

                    if detected_class:
                        x3d, y3d, z3d = estimate_3d_coordinates(x_center, y_center, frame_w, frame_h, detected_class)
                        timestamp = round(frame_idx / fps, 2)

                        point_data = {"x": x3d, "y": y3d, "z": z3d, "t": timestamp}
                        if detected_class == 'ball':
                            ball_points.append(point_data)
                        elif detected_class == 'bat':
                            bat_points.append(point_data)
                        elif detected_class == 'player':
                            player_points.append(point_data)
                        elif detected_class == 'stumps':
                            stump_points.append(point_data)

            frame_idx += 1
            if frame_idx > 150:  # Demo limit
                break

        cap.release()

        # Find last known positions
        final_batsman_pos = player_points[-1] if player_points else {"x": 0, "y": 0, "z": 0, "t": 0}
        final_bat_pos = bat_points[-1] if bat_points else {"x": 0, "y": 0, "z": 0, "t": 0}

        # Find unique stump positions
        distinct_stumps = []
        if stump_points:
            stump_df = pd.DataFrame(stump_points)
            distinct_stumps = stump_df.groupby(['x', 'y', 'z']).mean().reset_index().to_dict('records')[:3]

        # Output results
        result = {
            "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
            "video_info": {
                "filename": video_filename,
                "duration": round(frame_idx / fps, 2),
                "frames_processed": frame_idx
            },
            "ball_trajectory": ball_points,
            "bat_position": final_bat_pos,
            "batsman_leg_position": final_batsman_pos,
            "stump_coordinates": distinct_stumps
        }

        print("\n✅ Video Processed Successfully!")
        print(json.dumps(result, indent=2))

        # Visualize ball trajectory
        if ball_points:
            df = pd.DataFrame(ball_points)
            fig = plt.figure(figsize=(10, 6))
            ax = fig.add_subplot(111, projection='3d')

            ax.scatter(df['x'], df['y'], df['z'], c=df['t'], cmap='plasma')
            ax.set_xlabel('X (m)')
            ax.set_ylabel('Y (m)')
            ax.set_zlabel('Z (m)')
            ax.set_title('3D Ball Trajectory')
            plt.show()

        # Display uploaded video
        display(Video(temp_video_path, embed=True, width=600))

# Attach upload observer
video_upload.observe(handle_video_upload, names='value')


FileUpload(value={}, accept='.mp4,.avi,.mov', description='Upload Video')

FloatProgress(value=0.0, description='Processing:')

Output()